# SDH exp_006 — burden 2종 vs 3종 ablation

## 실험 질문

현재 최고 FE인 `mutation types + hotspot50`에 **여러 mutation token이 기록된 유전자 수**를 추가하면 OOF Macro F1이 개선되는가?

이 실험에서는 전처리 피처 한 개만 바꿉니다. Logistic Regression의 `C=1.0`, class weight, fold 수와 seed 등 공용 벤치마크 조건은 변경하지 않습니다.

## 1. 근거와 비교 설계

FE setting DB의 `pp-ij-B-burden3`은 gene burden 한 개 대비 3-seed 평균 Macro F1이 `+0.00456` 높았고 3개 seed 모두 양수였습니다.

- **case 01 — reference:** gene binary + gene/token burden + mutation types + fold-train hotspot 50
- **case 02 — burden3:** case 01 + `log1p(multi-mutated-gene count)`

`multi-mutated-gene count`는 환자 한 명에서 mutation token이 2개 이상 기록된 유전자 열의 개수입니다. 총 token 수가 같아도 변이가 여러 유전자에 흩어졌는지 특정 유전자에 집중됐는지를 구분하려는 피처입니다.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / 'common').is_dir()
        and (path / 'experiments').is_dir()
        and (path / 'data' / 'raw' / 'train.csv').is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        '저장소 안에서 Jupyter를 실행했는지와 data/raw/train.csv 존재 여부를 확인하세요.'
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin

from common.preprocessing_benchmark import run_preprocessing_benchmark
from experiments.SDH.exp_003_preprocessing.preprocessing import (
    MutationFeatureTransformer,
    _tokens,
)

TRAIN_PATH = PROJECT_ROOT / 'data' / 'raw' / 'train.csv'
RESULTS_DIR = PROJECT_ROOT / 'experiments' / 'SDH' / 'exp_006_burden3_ablation' / 'results'

print(f'프로젝트 루트: {PROJECT_ROOT}')
print(f'학습 데이터: {TRAIN_PATH}')

## 2. 원본 데이터 확인

원본은 읽기만 하며 수정하지 않습니다. ID 중복, target 결측, 5-fold가 불가능한 소수 클래스를 먼저 점검합니다.

In [ ]:
train = pd.read_csv(TRAIN_PATH)

assert {'ID', 'SUBCLASS'}.issubset(train.columns)
assert train['ID'].notna().all() and train['ID'].is_unique
assert train['SUBCLASS'].notna().all()
assert train['SUBCLASS'].value_counts().min() >= 5

print(f'행 수: {len(train):,}')
print(f'유전자 피처 수: {train.shape[1] - 2:,}')
print(f'암종 수: {train.SUBCLASS.nunique()}')
display(train[['ID', 'SUBCLASS']].head())

## 3. 세 번째 burden 구현

아래 구현이 reference와 candidate를 모두 만듭니다. 두 case의 유일한 차이는 `include_multi_mutated_gene_burden`입니다.

기본 transformer 내부의 hotspot 목록은 각 CV fold의 train 분할에서만 학습됩니다. 세 번째 burden은 각 환자 행만 보고 계산하므로 vocabulary나 target 통계를 학습하지 않습니다.

In [ ]:
class Burden3Transformer(TransformerMixin, BaseEstimator):
    def __init__(
        self,
        *,
        include_multi_mutated_gene_burden=False,
        hotspot_count=50,
        fill_value='WT',
        dtype='float32',
    ):
        self.include_multi_mutated_gene_burden = include_multi_mutated_gene_burden
        self.hotspot_count = hotspot_count
        self.fill_value = fill_value
        self.dtype = dtype

    def fit(self, X, y=None):
        self.base_transformer_ = MutationFeatureTransformer(
            min_gene_count=1,
            include_gene_burden=True,
            include_token_burden=True,
            include_mutation_type_counts=True,
            hotspot_count=self.hotspot_count,
            fill_value=self.fill_value,
            dtype=self.dtype,
        )
        self.base_transformer_.fit(X, y)
        self.feature_names_in_ = self.base_transformer_.feature_names_in_
        self.n_features_in_ = self.base_transformer_.n_features_in_
        return self

    def transform(self, X):
        if not hasattr(self, 'base_transformer_'):
            raise RuntimeError('transform 전에 fit을 실행해야 합니다.')

        output = self.base_transformer_.transform(X)
        if not self.include_multi_mutated_gene_burden:
            return output

        normalized = X.reindex(columns=self.feature_names_in_).fillna(self.fill_value)
        multi_gene_count = np.fromiter(
            (
                sum(len(_tokens(value, self.fill_value)) >= 2 for value in row)
                for row in normalized.to_numpy(dtype=object)
            ),
            dtype=np.int32,
            count=len(normalized),
        )
        output['log1p_multi_mutated_gene_burden'] = np.log1p(
            multi_gene_count
        ).astype(self.dtype)
        return output


candidates = {
    'case_01_two_burdens_reference': Burden3Transformer(
        include_multi_mutated_gene_burden=False,
    ),
    'case_02_three_burdens': Burden3Transformer(
        include_multi_mutated_gene_burden=True,
    ),
}

display(pd.DataFrame({
    name: transformer.get_params()
    for name, transformer in candidates.items()
}).T)

## 4. 피처 정의 sanity check

작은 행 몇 개에서 원래 문자열과 계산된 세 번째 burden을 나란히 확인합니다. 이 셀은 모델을 학습하지 않습니다.

In [ ]:
X = train.drop(columns=['ID', 'SUBCLASS'])
preview = X.head(5)

raw_multi_gene_count = preview.apply(
    lambda row: sum(len(_tokens(value, 'WT')) >= 2 for value in row),
    axis=1,
)
sanity = pd.DataFrame({
    'multi_mutated_gene_count': raw_multi_gene_count,
    'expected_log1p': np.log1p(raw_multi_gene_count),
})
display(sanity)

print('앞 200개 샘플의 multi-mutated-gene count 분포')
display(
    X.head(200).apply(
        lambda row: sum(len(_tokens(value, 'WT')) >= 2 for value in row),
        axis=1,
    ).describe().to_frame('count')
)

## 5. seed 42 1차 비교

공용 `run_preprocessing_benchmark()`를 그대로 사용합니다. 이 함수는 Logistic Regression의 `C`를 노출하지 않으며 기본 benchmark 값 `C=1.0`을 사용합니다.

In [ ]:
seed42_results = {}
seed42_summaries = []

for case_name, preprocessor in candidates.items():
    print(f'\n===== {case_name} =====')
    result = run_preprocessing_benchmark(
        train,
        preprocessor,
        experiment_id=f'exp_006_{case_name}',
        preprocessing_name=case_name,
        model='logistic',
        confirmation=False,
    )
    seed42_results[case_name] = result
    seed42_summaries.append(result.summary)

seed42_table = (
    pd.DataFrame(seed42_summaries)
    .sort_values('oof_f1_macro_mean', ascending=False)
    .reset_index(drop=True)
)
display(seed42_table[[
    'preprocessing', 'oof_f1_macro_mean', 'oof_accuracy_mean',
    'model_parameters', 'elapsed_seconds',
]])

## 6. seed 42 차이 확인

case 02에서 늘어난 피처 수가 정확히 1개인지 확인하고, Macro F1과 Accuracy 변화를 계산합니다.

In [ ]:
reference_name = 'case_01_two_burdens_reference'
candidate_name = 'case_02_three_burdens'

reference = seed42_results[reference_name]
candidate = seed42_results[candidate_name]
feature_counts = pd.DataFrame({
    reference_name: reference.fold_metrics['feature_count'].to_numpy(),
    candidate_name: candidate.fold_metrics['feature_count'].to_numpy(),
})
feature_counts['difference'] = (
    feature_counts[candidate_name] - feature_counts[reference_name]
)
assert feature_counts['difference'].eq(1).all()
display(feature_counts)

seed42_delta = (
    candidate.summary['oof_f1_macro_mean']
    - reference.summary['oof_f1_macro_mean']
)
accuracy_delta = (
    candidate.summary['oof_accuracy_mean']
    - reference.summary['oof_accuracy_mean']
)
print(f'Macro F1 변화: {seed42_delta:+.5f}')
print(f'Accuracy 변화: {accuracy_delta:+.5f}')

## 7. 3-seed confirmation

seed 42 결과와 관계없이 두 case를 모두 seed 42/52/62로 평가합니다. 동일 seed 안에서 case 02 − case 01의 paired 차이를 계산합니다.

In [ ]:
confirmation_results = {}
confirmation_summaries = []

for case_name, preprocessor in candidates.items():
    print(f'\n===== confirmation: {case_name} =====')
    result = run_preprocessing_benchmark(
        train,
        preprocessor,
        experiment_id=f'exp_006_{case_name}_confirmation',
        preprocessing_name=case_name,
        model='logistic',
        confirmation=True,
    )
    confirmation_results[case_name] = result
    confirmation_summaries.append(result.summary)

confirmation_table = (
    pd.DataFrame(confirmation_summaries)
    .sort_values('oof_f1_macro_mean', ascending=False)
    .reset_index(drop=True)
)
display(confirmation_table[[
    'preprocessing', 'oof_f1_macro_mean', 'oof_f1_macro_std',
    'oof_accuracy_mean', 'model_parameters',
]])

## 8. seed별 paired 판정과 저장

평균이 높더라도 한 seed에만 의존하면 채택하지 않습니다. 세 seed 모두 양수인지와 평균 증분의 크기를 함께 봅니다. 저장 대상은 lightweight metrics와 leaderboard뿐이며 OOF 예측·확률은 저장하지 않습니다.

In [ ]:
reference_runs = confirmation_results[reference_name].run_metrics.set_index('cv_seed')
candidate_runs = confirmation_results[candidate_name].run_metrics.set_index('cv_seed')
paired = pd.DataFrame({
    'reference_f1': reference_runs['oof_f1_macro'],
    'burden3_f1': candidate_runs['oof_f1_macro'],
})
paired['delta'] = paired['burden3_f1'] - paired['reference_f1']
display(paired.style.format('{:.5f}'))

mean_delta = float(paired['delta'].mean())
all_positive = bool(paired['delta'].gt(0).all())
decision = '채택 후보' if mean_delta > 0 and all_positive else '보류/기각'
print(f'3-seed 평균 paired 변화: {mean_delta:+.5f}')
print(f'3/3 seed 양수: {all_positive}')
print(f'판정: {decision}')

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for case_name, result in seed42_results.items():
    result.save_metrics(RESULTS_DIR / f'metrics_{case_name}.json')
for case_name, result in confirmation_results.items():
    result.save_metrics(RESULTS_DIR / f'metrics_{case_name}_confirmation.json')
seed42_table.to_csv(RESULTS_DIR / 'leaderboard.csv', index=False)
confirmation_table.to_csv(RESULTS_DIR / 'leaderboard_confirmation.csv', index=False)
print(f'저장 완료: {RESULTS_DIR}')

## 최종 해석 체크리스트

실행 후 `EXPERIMENT_SUMMARY.md`에 다음을 기록합니다.

1. seed 42 Macro F1 및 Accuracy 변화
2. 두 case의 3-seed 평균 ± 표준편차
3. seed 42/52/62 각각의 paired 변화
4. 세 번째 burden 한 개 외에 피처 수 차이가 없는지
5. 채택 또는 기각 결정

DB의 기존 `+0.00456`은 다른 파이프라인과 `C=0.1`에서 얻은 근거이므로, SDH 공용 `C=1.0` 결과가 재현되지 않아도 모순은 아닙니다.